# S5-02: MCP 서버 심화 — Resources, Prompts, Client
**Skilljar L07-L10: Resources / Accessing Resources / Prompts / Prompts in the Client**

## 학습 목표
- `@mcp.resource()` 로 정적/동적 리소스를 정의한다
- `@mcp.prompt()` 로 재사용 가능한 프롬프트 템플릿을 정의한다
- MCP 클라이언트로 서버에 연결하여 도구, 리소스, 프롬프트를 사용한다
- 건축 법규 데이터를 MCP 리소스로 제공하는 서버를 구현한다

## 사전 준비
1. S5-01 완료 (FastMCP 설치 확인)
2. `.env` 파일에 `ANTHROPIC_API_KEY` 설정

In [ ]:
# 패키지 설치 확인
%pip install "mcp[cli]" anthropic python-dotenv pydantic

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## 1. MCP 리소스 개념

MCP 서버의 3가지 Primitive 비교:

| Primitive | 동작 | 부작용 | 예시 |
| --- | --- | --- | --- |
| **Tools** | 함수 **실행** | 있을 수 있음 | 계산, API 호출, 파일 쓰기 |
| **Resources** | 데이터 **읽기** | 없음 (순수 읽기) | KDS 조문, 철근 DB, BIM 데이터 |
| **Prompts** | 템플릿 **생성** | 없음 | 구조 검토 프롬프트, 보고서 양식 |

리소스의 두 가지 유형:
- **정적 리소스**: URI가 고정 (예: `kds://14-20-20/general`)
- **동적 리소스 (템플릿)**: URI에 매개변수 포함 (예: `rebar://{designation}`)

In [ ]:
# 리소스 정의 패턴 시연
from mcp.server.fastmcp import FastMCP

demo_mcp = FastMCP("resource-demo")

# 정적 리소스 — URI가 고정
@demo_mcp.resource("config://app/version")
def get_version() -> str:
    """앱 버전 정보를 반환합니다."""
    return "v1.0.0 — KDS Structural Review Server"

# 동적 리소스 (템플릿) — URI에 매개변수
@demo_mcp.resource("greeting://{language}")
def get_greeting(language: str) -> str:
    """언어별 인사말을 반환합니다."""
    greetings = {
        "ko": "안녕하세요!",
        "en": "Hello!",
        "ja": "こんにちは!"
    }
    return greetings.get(language, f"Unknown language: {language}")

# 직접 호출 테스트
print("정적 리소스:", get_version())
print("동적 리소스 (ko):", get_greeting("ko"))
print("동적 리소스 (en):", get_greeting("en"))

## 2. MCP 프롬프트 개념

프롬프트는 **재사용 가능한 LLM 지시사항 템플릿**이다.

활용 예시:
- 구조 검토 프롬프트: 부재 유형별 검토 항목이 다름
- 보고서 작성 프롬프트: 출력 형식별 (표, 서술형, JSON) 템플릿
- 코드 리뷰 프롬프트: 언어별/프레임워크별 검토 기준

In [ ]:
# 프롬프트 정의 패턴 시연
from mcp.server.fastmcp import FastMCP

prompt_mcp = FastMCP("prompt-demo")

@prompt_mcp.prompt()
def code_review(language: str, focus: str = "general") -> str:
    """코드 리뷰 프롬프트 템플릿

    Args:
        language: 프로그래밍 언어 (python, javascript, etc)
        focus: 검토 초점 (general, security, performance)
    """
    focus_guides = {
        "general": "코드 가독성, 구조, 네이밍, 에러 처리",
        "security": "입력 검증, 인증, 인가, 데이터 보호",
        "performance": "시간 복잡도, 메모리 사용, 캐싱 전략"
    }

    return (
        f"다음 {language} 코드를 리뷰하세요.\n\n"
        f"검토 초점: {focus_guides.get(focus, focus_guides['general'])}\n\n"
        f"리뷰 형식:\n"
        f"1. 요약 (1-2문장)\n"
        f"2. 장점\n"
        f"3. 개선 사항 (구체적 코드 위치 명시)\n"
        f"4. 심각도 (Critical / Major / Minor)"
    )

# 직접 호출 테스트
print("=== Python 일반 리뷰 ===")
print(code_review("python", "general"))
print("\n=== JavaScript 보안 리뷰 ===")
print(code_review("javascript", "security"))

---
## 연습 1: KDS 법규 리소스 서버 구현

### 문제
KDS 건축구조기준 데이터를 MCP 리소스로 제공하는 서버를 구현하세요.

**요구사항:**
1. 정적 리소스: `kds://summary` — 전체 KDS 기준 목록 반환
2. 동적 리소스: `kds://{code}/info` — 특정 KDS 코드의 상세 정보 반환
3. 동적 리소스: `kds://{code}/{section}` — 특정 조항 내용 반환

**KDS 데이터 (아래 딕셔너리 사용):**
```python
KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16"
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min"
        }
    }
}
```

**기대 출력:**
```
kds://summary → "사용 가능한 KDS 기준: 14-20-20 (콘크리트구조 일반설계기준), ..."
kds://14-20-20/info → "KDS 14 20 20 콘크리트구조 일반설계기준\n조항: 4.2.3, 4.3.2"
kds://14-20-20/4.2.3 → "설계휨강도: phi*Mn >= Mu"
```

In [ ]:
# TODO: 아래 코드를 완성하세요

from mcp.server.fastmcp import FastMCP

kds_mcp = FastMCP("kds-resource-server")

KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16"
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min"
        }
    }
}

# TODO: 3개의 리소스를 정의하세요
# 1. @kds_mcp.resource("kds://summary")
# 2. @kds_mcp.resource("kds://{code}/info")
# 3. @kds_mcp.resource("kds://{code}/{section}")

In [ ]:
# === 솔루션 ===

from mcp.server.fastmcp import FastMCP

kds_mcp = FastMCP("kds-resource-server")

KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16"
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min"
        }
    }
}

@kds_mcp.resource("kds://summary")
def get_kds_summary() -> str:
    """사용 가능한 KDS 기준 목록을 반환합니다."""
    items = [f"{code} ({info['title']})" for code, info in KDS_DATA.items()]
    return "사용 가능한 KDS 기준: " + ", ".join(items)

@kds_mcp.resource("kds://{code}/info")
def get_kds_info(code: str) -> str:
    """특정 KDS 코드의 상세 정보를 반환합니다.

    Args:
        code: KDS 코드 (예: 14-20-20)
    """
    if code not in KDS_DATA:
        return f"알 수 없는 KDS 코드: {code}"
    kds = KDS_DATA[code]
    sections = ", ".join(kds["sections"].keys())
    return f"KDS {code} {kds['title']}\n조항: {sections}"

@kds_mcp.resource("kds://{code}/{section}")
def get_kds_section(code: str, section: str) -> str:
    """특정 KDS 조항의 내용을 반환합니다.

    Args:
        code: KDS 코드
        section: 조항 번호
    """
    if code not in KDS_DATA:
        return f"알 수 없는 KDS 코드: {code}"
    sections = KDS_DATA[code]["sections"]
    if section not in sections:
        return f"조항 {section}이 없습니다. 사용 가능: {', '.join(sections.keys())}"
    return sections[section]

# 테스트
print("=== kds://summary ===")
print(get_kds_summary())
print("\n=== kds://14-20-20/info ===")
print(get_kds_info("14-20-20"))
print("\n=== kds://14-20-20/4.2.3 ===")
print(get_kds_section("14-20-20", "4.2.3"))

In [ ]:
# 검증 함수
def verify_exercise1():
    passed = 0
    tests = [
        ("14-20-20" in get_kds_summary(), "summary에 14-20-20 포함"),
        ("14-20-22" in get_kds_summary(), "summary에 14-20-22 포함"),
        ("콘크리트" in get_kds_info("14-20-20"), "info에 제목 포함"),
        ("4.2.3" in get_kds_info("14-20-20"), "info에 조항 번호 포함"),
        ("phi*Mn" in get_kds_section("14-20-20", "4.2.3"), "조항 내용 정확"),
        ("알 수 없는" in get_kds_section("99-99-99", "1.1"), "없는 코드 처리"),
    ]

    for check, name in tests:
        if check:
            print(f"  PASS: {name}")
            passed += 1
        else:
            print(f"  FAIL: {name}")

    print(f"\n결과: {passed}/{len(tests)} 통과")
    return passed == len(tests)

verify_exercise1()

---
## 연습 2: 구조 검토 프롬프트 템플릿 서버

### 문제
다양한 구조 부재에 대한 검토 프롬프트 템플릿을 제공하는 MCP 서버를 구현하세요.

**요구사항:**
1. `@mcp.prompt()` 로 `structural_review(member_type, output_format)` 정의
   - `member_type`: "beam", "column", "slab" 중 택1
   - `output_format`: "table" (기본), "json", "narrative"
2. 부재별 검토 항목이 다르게 포함
3. 출력 형식 지시사항 포함

**기대 출력 (beam, table):**
```
RC 보의 설계 적정성을 검토하세요.

검토 항목:
1. 휨 강도 (KDS 14 20 20)
2. 전단 강도 (KDS 14 20 22)
3. 처짐 검토

출력 형식: 결과를 마크다운 표로 정리하세요.
```

In [ ]:
# TODO: 아래 코드를 완성하세요

from mcp.server.fastmcp import FastMCP

prompt_mcp = FastMCP("structural-prompt-server")

@prompt_mcp.prompt()
def structural_review(member_type: str, output_format: str = "table") -> str:
    """구조 부재 검토 프롬프트 템플릿

    Args:
        member_type: 부재 유형 (beam, column, slab)
        output_format: 출력 형식 (table, json, narrative)
    """
    # TODO: 부재별 검토 항목 + 출력 형식 지시사항을 반환
    pass

In [ ]:
# === 솔루션 ===

from mcp.server.fastmcp import FastMCP

prompt_mcp = FastMCP("structural-prompt-server")

@prompt_mcp.prompt()
def structural_review(member_type: str, output_format: str = "table") -> str:
    """구조 부재 검토 프롬프트 템플릿

    Args:
        member_type: 부재 유형 (beam, column, slab)
        output_format: 출력 형식 (table, json, narrative)
    """
    review_items = {
        "beam": [
            "1. 휨 강도 (KDS 14 20 20)",
            "2. 전단 강도 (KDS 14 20 22)",
            "3. 처짐 검토"
        ],
        "column": [
            "1. 축력비 검토 (최대 0.8)",
            "2. 철근비 검토 (0.01~0.08)",
            "3. 띠철근 간격 검토"
        ],
        "slab": [
            "1. 최소 두께 검토 (KDS 14 20 20)",
            "2. 배근 간격 검토",
            "3. 펀칭 전단 검토"
        ]
    }

    format_instructions = {
        "table": "결과를 마크다운 표로 정리하세요.",
        "json": "결과를 JSON 형식으로 출력하세요.",
        "narrative": "설계 근거와 함께 서술형으로 작성하세요."
    }

    member_names = {"beam": "보", "column": "기둥", "slab": "슬래브"}

    if member_type.lower() not in review_items:
        return f"지원하는 부재 유형: {', '.join(review_items.keys())}"

    name = member_names[member_type.lower()]
    items = "\n".join(review_items[member_type.lower()])
    fmt = format_instructions.get(output_format, format_instructions["table"])

    return (
        f"RC {name}의 설계 적정성을 검토하세요.\n\n"
        f"검토 항목:\n{items}\n\n"
        f"출력 형식: {fmt}"
    )

# 테스트
print("=== beam, table ===")
print(structural_review("beam", "table"))
print("\n=== column, json ===")
print(structural_review("column", "json"))
print("\n=== slab, narrative ===")
print(structural_review("slab", "narrative"))

In [ ]:
# 검증 함수
def verify_exercise2():
    passed = 0
    tests = [
        ("보" in structural_review("beam"), "beam → '보' 포함"),
        ("기둥" in structural_review("column"), "column → '기둥' 포함"),
        ("슬래브" in structural_review("slab"), "slab → '슬래브' 포함"),
        ("표" in structural_review("beam", "table"), "table 형식 지시"),
        ("JSON" in structural_review("beam", "json"), "json 형식 지시"),
        ("서술" in structural_review("beam", "narrative"), "narrative 형식 지시"),
        ("KDS" in structural_review("beam"), "KDS 기준 참조"),
    ]

    for check, name in tests:
        if check:
            print(f"  PASS: {name}")
            passed += 1
        else:
            print(f"  FAIL: {name}")

    print(f"\n결과: {passed}/{len(tests)} 통과")
    return passed == len(tests)

verify_exercise2()

---
## 연습 3: 통합 서버 파일 작성 (Tools + Resources + Prompts)

### 문제
앞의 연습 1 (리소스)과 연습 2 (프롬프트)에 도구를 추가하여 **통합 MCP 서버**를 `kds_server.py`로 작성하세요.

**요구사항:**
1. 리소스 2개: `kds://summary`, `kds://{code}/info`
2. 도구 1개: `search_kds(keyword: str)` — 키워드로 KDS 조항 검색
3. 프롬프트 1개: `structural_review(member_type: str)` — 부재별 검토 프롬프트
4. `mcp.run()` 포함, Inspector로 테스트 가능

**기대 동작:**
```
Inspector Tools 탭: search_kds("전단") → [{"code": "14-20-22", ...}]
Inspector Resources 탭: kds://summary → 기준 목록
Inspector Prompts 탭: structural_review("beam") → 검토 프롬프트
```

In [ ]:
# TODO: kds_server.py 파일 내용을 작성하세요

server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("kds-integrated-server")

KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16"
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min"
        }
    }
}

# TODO: 리소스 2개 + 도구 1개 + 프롬프트 1개 정의

if __name__ == "__main__":
    mcp.run()
'''

with open("kds_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("kds_server.py 저장 완료")
print("테스트: 터미널에서 'mcp dev kds_server.py' 실행")

In [ ]:
# === 솔루션 ===

server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("kds-integrated-server")

KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16"
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min"
        }
    }
}

# === 리소스 ===

@mcp.resource("kds://summary")
def get_kds_summary() -> str:
    """사용 가능한 KDS 기준 목록을 반환합니다."""
    items = [f"{code} ({info[\'title\'])" for code, info in KDS_DATA.items()]
    return "사용 가능한 KDS 기준: " + ", ".join(items)

@mcp.resource("kds://{code}/info")
def get_kds_info(code: str) -> str:
    """특정 KDS 코드의 상세 정보를 반환합니다."""
    if code not in KDS_DATA:
        return f"알 수 없는 KDS 코드: {code}"
    kds = KDS_DATA[code]
    sections = ", ".join(kds["sections"].keys())
    return f"KDS {code} {kds[\'title\']}\\n조항: {sections}"

# === 도구 ===

@mcp.tool()
def search_kds(keyword: str) -> dict:
    """KDS 건축구조기준에서 키워드로 조항을 검색합니다.

    Args:
        keyword: 검색 키워드
    """
    results = []
    for code, kds in KDS_DATA.items():
        for section, content in kds["sections"].items():
            if keyword.lower() in content.lower() or keyword.lower() in kds["title"].lower():
                results.append({
                    "code": code,
                    "section": section,
                    "content": content
                })
    return {"keyword": keyword, "count": len(results), "results": results}

# === 프롬프트 ===

@mcp.prompt()
def structural_review(member_type: str) -> str:
    """구조 부재 검토 프롬프트 템플릿

    Args:
        member_type: 부재 유형 (beam, column)
    """
    templates = {
        "beam": "RC 보의 설계를 검토하세요.\\n1. 휨 강도 (KDS 14 20 20)\\n2. 전단 강도 (KDS 14 20 22)\\n3. 처짐",
        "column": "RC 기둥의 설계를 검토하세요.\\n1. 축력비\\n2. 철근비\\n3. 띠철근"
    }
    return templates.get(member_type.lower(), "지원: beam, column")

if __name__ == "__main__":
    mcp.run()
'''

with open("kds_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("kds_server.py 저장 완료")
print("\n테스트: 터미널에서 'mcp dev kds_server.py' 실행 후")
print("  - Tools 탭: search_kds 호출")
print("  - Resources 탭: kds://summary 읽기")
print("  - Prompts 탭: structural_review 호출")

In [ ]:
# 검증 함수
import os

def verify_exercise3():
    if not os.path.exists("kds_server.py"):
        print("  FAIL: kds_server.py 파일이 없습니다")
        return False

    with open("kds_server.py", "r", encoding="utf-8") as f:
        content = f.read()

    checks = [
        ("FastMCP" in content, "FastMCP 사용"),
        ("@mcp.resource" in content, "리소스 정의"),
        ("@mcp.tool" in content, "도구 정의"),
        ("@mcp.prompt" in content, "프롬프트 정의"),
        ("search_kds" in content, "검색 도구"),
        ("structural_review" in content, "검토 프롬프트"),
        ("kds://summary" in content or "kds://" in content, "리소스 URI"),
        ("mcp.run()" in content, "서버 실행"),
    ]

    passed = 0
    for check, name in checks:
        if check:
            print(f"  PASS: {name}")
            passed += 1
        else:
            print(f"  FAIL: {name}")

    print(f"\n결과: {passed}/{len(checks)} 통과")
    return passed == len(checks)

verify_exercise3()

---
## 건축공학 실습: 철근 데이터베이스 MCP 리소스

### 과제
철근 규격 정보를 MCP 리소스로 제공하고, 철근 배근 계산을 도구로 제공하는 서버를 구현하세요.

**요구사항:**
1. 정적 리소스: `rebar://catalog` — 전체 철근 목록
2. 동적 리소스: `rebar://{designation}` — 개별 철근 상세 정보
3. 도구: `recommend_rebar(As_required: float, max_bars: int = 8)` — 소요 철근량에 맞는 배근 추천
4. 프롬프트: `rebar_report()` — 배근 검토 보고서 프롬프트

**철근 데이터:**
```python
REBAR_DB = {
    "D10": {"diameter": 9.53, "area": 71.33, "weight": 0.560},
    "D13": {"diameter": 12.7, "area": 126.7, "weight": 0.995},
    "D16": {"diameter": 15.9, "area": 198.6, "weight": 1.56},
    "D19": {"diameter": 19.1, "area": 286.5, "weight": 2.25},
    "D22": {"diameter": 22.2, "area": 387.1, "weight": 3.04},
    "D25": {"diameter": 25.4, "area": 506.7, "weight": 3.98},
    "D29": {"diameter": 28.6, "area": 642.4, "weight": 5.04},
    "D32": {"diameter": 31.8, "area": 794.2, "weight": 6.23},
}
```

**기대 출력:**
```
recommend_rebar(1500, 6) → {"options": [{"rebar": "D19", "count": 6, "total": 1719}, ...]}
```

In [ ]:
# TODO: 아래 코드를 완성하세요

import math
from mcp.server.fastmcp import FastMCP

rebar_mcp = FastMCP("rebar-server")

REBAR_DB = {
    "D10": {"diameter": 9.53, "area": 71.33, "weight": 0.560},
    "D13": {"diameter": 12.7, "area": 126.7, "weight": 0.995},
    "D16": {"diameter": 15.9, "area": 198.6, "weight": 1.56},
    "D19": {"diameter": 19.1, "area": 286.5, "weight": 2.25},
    "D22": {"diameter": 22.2, "area": 387.1, "weight": 3.04},
    "D25": {"diameter": 25.4, "area": 506.7, "weight": 3.98},
    "D29": {"diameter": 28.6, "area": 642.4, "weight": 5.04},
    "D32": {"diameter": 31.8, "area": 794.2, "weight": 6.23},
}

# TODO: 리소스 2개 + 도구 1개 + 프롬프트 1개 정의

In [ ]:
# === 솔루션 ===

import math
from mcp.server.fastmcp import FastMCP

rebar_mcp = FastMCP("rebar-server")

REBAR_DB = {
    "D10": {"diameter": 9.53, "area": 71.33, "weight": 0.560},
    "D13": {"diameter": 12.7, "area": 126.7, "weight": 0.995},
    "D16": {"diameter": 15.9, "area": 198.6, "weight": 1.56},
    "D19": {"diameter": 19.1, "area": 286.5, "weight": 2.25},
    "D22": {"diameter": 22.2, "area": 387.1, "weight": 3.04},
    "D25": {"diameter": 25.4, "area": 506.7, "weight": 3.98},
    "D29": {"diameter": 28.6, "area": 642.4, "weight": 5.04},
    "D32": {"diameter": 31.8, "area": 794.2, "weight": 6.23},
}

# === 리소스 ===

@rebar_mcp.resource("rebar://catalog")
def get_rebar_catalog() -> str:
    """전체 철근 규격 목록을 반환합니다."""
    lines = ["철근 규격 카탈로그 (KS D 3504):"]
    for name, info in REBAR_DB.items():
        lines.append(
            f"  {name}: 직경 {info['diameter']}mm, "
            f"단면적 {info['area']}mm2, "
            f"단위무게 {info['weight']}kg/m"
        )
    return "\n".join(lines)

@rebar_mcp.resource("rebar://{designation}")
def get_rebar_detail(designation: str) -> str:
    """개별 철근의 상세 정보를 반환합니다.

    Args:
        designation: 철근 호칭 (예: D25)
    """
    key = designation.upper()
    if key not in REBAR_DB:
        return f"알 수 없는 철근: {designation}. 사용 가능: {', '.join(REBAR_DB.keys())}"
    info = REBAR_DB[key]
    return (
        f"철근 {key} 상세 정보 (KS D 3504)\n"
        f"  공칭 직경: {info['diameter']} mm\n"
        f"  공칭 단면적: {info['area']} mm2\n"
        f"  단위 무게: {info['weight']} kg/m"
    )

# === 도구 ===

@rebar_mcp.tool()
def recommend_rebar(As_required: float, max_bars: int = 8) -> dict:
    """소요 철근량에 맞는 배근 조합을 추천합니다.

    Args:
        As_required: 소요 철근량 (mm2)
        max_bars: 최대 철근 개수 (기본: 8)
    """
    options = []
    for name, info in sorted(REBAR_DB.items(), key=lambda x: x[1]["area"]):
        count = math.ceil(As_required / info["area"])
        if count <= max_bars:
            total = round(count * info["area"], 1)
            ratio = round(total / As_required, 3)
            options.append({
                "rebar": name,
                "count": count,
                "total_area_mm2": total,
                "utilization_ratio": ratio
            })

    return {
        "As_required_mm2": As_required,
        "max_bars": max_bars,
        "option_count": len(options),
        "options": options
    }

# === 프롬프트 ===

@rebar_mcp.prompt()
def rebar_report() -> str:
    """배근 검토 보고서 프롬프트 템플릿"""
    return (
        "배근 검토 보고서를 작성하세요.\n\n"
        "1. rebar://catalog에서 사용 가능한 철근 규격을 확인하세요.\n"
        "2. recommend_rebar 도구로 배근 조합을 산출하세요.\n"
        "3. 다음 항목을 포함하여 보고서를 작성하세요:\n"
        "   - 소요 철근량 (As,req)\n"
        "   - 배근 조합 3가지 (경제성 비교)\n"
        "   - 추천 배근 및 근거\n"
        "   - 결과 요약표"
    )

# 테스트
print("=== 카탈로그 ===")
print(get_rebar_catalog())
print("\n=== D25 상세 ===")
print(get_rebar_detail("D25"))
print("\n=== 배근 추천 (As=1500) ===")
result = recommend_rebar(1500, 6)
for opt in result["options"]:
    print(f"  {opt['count']}-{opt['rebar']}: {opt['total_area_mm2']}mm2 (비율: {opt['utilization_ratio']})")
print("\n=== 보고서 프롬프트 ===")
print(rebar_report())

In [ ]:
# 검증 함수
def verify_structural():
    passed = 0

    # 카탈로그 리소스
    catalog = get_rebar_catalog()
    if "D25" in catalog and "D10" in catalog:
        print("  PASS: 카탈로그에 철근 포함")
        passed += 1
    else:
        print("  FAIL: 카탈로그 데이터 누락")

    # 상세 리소스
    detail = get_rebar_detail("D25")
    if "506.7" in detail:
        print("  PASS: D25 단면적 정확")
        passed += 1
    else:
        print("  FAIL: D25 단면적 오류")

    # 배근 추천 도구
    result = recommend_rebar(1500, 6)
    if result["option_count"] > 0:
        print(f"  PASS: 배근 옵션 {result['option_count']}개 생성")
        passed += 1
    else:
        print("  FAIL: 배근 옵션 없음")

    # 모든 옵션이 소요량 이상인지 확인
    all_sufficient = all(
        opt["total_area_mm2"] >= 1500 for opt in result["options"]
    )
    if all_sufficient:
        print("  PASS: 모든 옵션이 소요량 충족")
        passed += 1
    else:
        print("  FAIL: 소요량 미달 옵션 존재")

    # 프롬프트
    prompt = rebar_report()
    if "recommend_rebar" in prompt and "보고서" in prompt:
        print("  PASS: 프롬프트 내용 적절")
        passed += 1
    else:
        print("  FAIL: 프롬프트 내용 부족")

    print(f"\n결과: {passed}/5 통과")
    return passed == 5

verify_structural()